In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 31.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import json as _json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from pathlib import Path as _P
import time as _time

SEED = 42
DATA_PATH = _P("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 100
PATIENCE = 15
TEST_YEAR_CUTOFF = 2023
N_SPLITS = 5
OPTUNA_TRIALS = 15
N_FINAL_SEEDS = 5 
N_FOLDS_FV = 5 
WINSOR_CAP = 3.0
SYMLOG_CAP = float(np.log1p(WINSOR_CAP))

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## Data Processing

In [3]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    df = pd.read_csv(data_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["Close_logret"] = df.groupby("Company")["Close"].transform(lambda x: np.log(x / x.shift(1)))
    df["Volume_log"] = np.log1p(df["Volume"].clip(lower=0))

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()

    prior_static_cols = [
        "Prior_EBITDA_SL", "Prior_NI_SL", "Prior_ROA",
        "Leverage", "Cash_Ratio", "Size_SL",
        "Prior_Net_Sales_SL", "Prior_OpEx_SL", "Prior_FCF_Per_Share_SL",
    ]
    current_year_cols = [
        "Total_Assets", "Total_Debt", "Cash",
        "Net_Sales", "Operating_Expenses", "FCF_Per_Share",
    ]
    static_input_cols = ["MC_MIB", "MC_MID", "MC_SMALL", "Sector"]
    exclude_cols = ({"Date", "Company", "year", "has_targets"}
                    | set(PREDICTION_TARGETS) | set(prior_static_cols)
                    | set(current_year_cols) | set(static_input_cols))
    feature_cols = [col for col in df.columns
                    if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company)
                & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[
                (company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)
            ].copy()

            if len(window) < 200:
                continue

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            raw_window = window[feature_cols].to_numpy(dtype=np.float32)

            static_data = window[prior_static_cols].iloc[-1].to_numpy(dtype=np.float32)
            sector_str = str(window["Sector"].iloc[-1])

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": raw_window,
                "static_data": static_data,
                "sector": sector_str,
                "window_dates": window["Date"].to_numpy(),
                "window_days": (window["Date"] - window["Date"].iloc[0]).dt.days.to_numpy(),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)

            data_records.append({
                "company": seq["company"], "year": seq["year"],
                "year_end_date": seq["year_end_date"],
                "target": target_name, "label_value": label_value,
                "window_data": seq["window_data"],
                "static_data": seq["static_data"],
                "sector": seq["sector"],
                "window_dates": seq["window_dates"],
                "window_days": seq["window_days"],
            })

    return pd.DataFrame(data_records), feature_cols

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)
print(f"Total rows: {len(data_df)}")
print(f"feature_cols ({len(feature_cols)}): {feature_cols}")

Total rows: 6715
feature_cols (34): ['Close', 'High', 'Low', 'Open', 'Volume', 'DE10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume', 'Euribor_3M', 'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'IT10YT_Yield', 'IT_Inflation', 'IT_GDP', 'IT_Unemployment', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'TTF_Gas_Close', 'VIX_Close', 'Gold_Close', 'T5YIE', 'YC__level', 'YC__slope', 'YC__curv', 'PCOPPUSDM', 'PALUMUSDM', 'NFCI', 'ff_Mkt-RF', 'ff_HML', 'ff_Mom', 'Close_logret', 'Volume_log']


## Pipeline Setup (Symlog, Winsor +/-3, Sector + Prior Statics, 5-Fold TSCV)

In [4]:
def temporal_kfold_split(df: pd.DataFrame, n_splits: int = N_SPLITS, test_year_cutoff: int = TEST_YEAR_CUTOFF):
    if df.empty:
        raise ValueError("The dataset is empty.")
    available_years = sorted(int(y) for y in df["year"].unique())
    pre_test_years = [y for y in available_years if y <= test_year_cutoff]
    print(f"Available years: {available_years}")
    if len(pre_test_years) < n_splits + 1:
        raise ValueError(f"Not enough pre-test years ({len(pre_test_years)}) for {n_splits} folds.")
    val_years = pre_test_years[-n_splits:]
    test_set = df[df["year"] > test_year_cutoff].copy()
    print(f"Test: > {test_year_cutoff} ({len(test_set)} samples)")
    print(f"Temporal k-fold ({n_splits} folds, expanding window). Val years: {val_years}")
    folds = []
    for fold_idx, vy in enumerate(val_years, start=1):
        train_set = df[df["year"] < vy].copy()
        val_set = df[df["year"] == vy].copy()
        if train_set.empty or val_set.empty:
            print(f"WARNING: fold {fold_idx} (val {vy}) is empty. Skipping.")
            continue
        folds.append((train_set, val_set, vy, fold_idx))
        print(f"  Fold {fold_idx}: train <= {vy-1} ({len(train_set)}), val = {vy} ({len(val_set)})")
    if not folds:
        raise ValueError("No valid folds produced.")
    return folds, test_set

def split_to_single(df: pd.DataFrame, test_year_cutoff: int = TEST_YEAR_CUTOFF):
    folds, test_set = temporal_kfold_split(df, n_splits=1, test_year_cutoff=test_year_cutoff)
    train_set, val_set, _, _ = folds[-1]
    return train_set, val_set, test_set

def make_robust_scaler(train_df: pd.DataFrame, feature_indices):
    sc = RobustScaler()
    train_windows = np.vstack([row[:, feature_indices] for row in train_df["window_data"]])
    sc.fit(train_windows)
    return sc

def fit_static_scaler(mtl_df, new_slice):
    arr = np.vstack(mtl_df["static_data"].values).astype(np.float32)
    return RobustScaler().fit(arr[:, new_slice])

def apply_static_scaler(mtl_df, sc, new_slice):
    df = mtl_df.copy().reset_index(drop=True)
    arr = np.vstack(df["static_data"].values).astype(np.float32)
    arr[:, new_slice] = sc.transform(arr[:, new_slice]).astype(np.float32)
    df["static_data"] = [arr[i] for i in range(len(arr))]
    return df

SECTOR_LIST = sorted(data_df["sector"].dropna().unique())
SECTOR_TO_IDX = {s: i for i, s in enumerate(SECTOR_LIST)}
N_SECTORS = len(SECTOR_LIST)
print(f"Sectors ({N_SECTORS}): {SECTOR_LIST}")

def sector_onehot(sector_str):
    v = np.zeros(N_SECTORS, dtype=np.float32)
    if sector_str in SECTOR_TO_IDX:
        v[SECTOR_TO_IDX[sector_str]] = 1.0
    return v

def build_full_static(static_data_arr, sector_str):
    prior = np.asarray(static_data_arr, dtype=np.float32)  # 9 dims
    sector_oh = sector_onehot(sector_str)                   # N_SECTORS dims
    return np.concatenate([prior, sector_oh]).astype(np.float32)  # 9 + N_SECTORS

data_df["static_data"] = data_df.apply(
    lambda r: build_full_static(r["static_data"], r["sector"]), axis=1
)
STATIC_DIM_IMP = 9 + N_SECTORS  # 9 prior + N_SECTORS sector one-hot
NEW_STATIC_SLICE = slice(0, 9)  # scale only the prior (first 9) dims
print(f"STATIC_DIM_IMP={STATIC_DIM_IMP} (prior=9, sector={N_SECTORS})")

def extract_mtl_df(df, new_slice=NEW_STATIC_SLICE):
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        row = group.iloc[0]
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": row["year_end_date"],
            "window_data": row["window_data"],
            "static_data": row["static_data"],
            "window_dates": row["window_dates"],
            "window_days": row["window_days"],
            "label_value": label_value,
        })
    return pd.DataFrame(mtl_records)

# Apply winsor +/-3 + symlog to YoY ratio
sym_lab = data_df["label_value"].to_numpy()
raw_ratio = np.sign(sym_lab) * np.expm1(np.abs(sym_lab))
capped = np.clip(raw_ratio, -WINSOR_CAP, WINSOR_CAP)
data_df["label_value"] = (np.sign(capped) * np.log1p(np.abs(capped))).astype(np.float32)
n_capped = int(np.sum(np.abs(raw_ratio) > WINSOR_CAP))
print(f"Winsor cap: {n_capped}/{len(raw_ratio)} ({100.0*n_capped/len(raw_ratio):.2f}%) at +/-{WINSOR_CAP}")

# Build k-fold for HPO + single-split for final test
folds, test_data = temporal_kfold_split(data_df, n_splits=N_SPLITS, test_year_cutoff=TEST_YEAR_CUTOFF)
train_data, val_data, _ = split_to_single(data_df, test_year_cutoff=TEST_YEAR_CUTOFF)

mtl_train_data_raw = extract_mtl_df(train_data)
mtl_val_data_raw   = extract_mtl_df(val_data)
mtl_test_data      = extract_mtl_df(test_data)

train_per_year = mtl_train_data_raw["year"].value_counts().sort_index()
test_per_year  = mtl_test_data["year"].value_counts().sort_index()
print(f"Train: {len(mtl_train_data_raw)} ({dict(train_per_year)})")
print(f"Test:  {len(mtl_test_data)} ({dict(test_per_year)})")

print("Pipeline setup complete.")

Sectors (10): ['Basic Materials', 'Consumer Cyclicals', 'Consumer Non-Cyclicals', 'Energy', 'Financials', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Utilities']
STATIC_DIM_IMP=19 (prior=9, sector=10)
Winsor cap: 1216/6715 (18.11%) at +/-3.0
Available years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Test: > 2023 (1028 samples)
Temporal k-fold (5 folds, expanding window). Val years: [2019, 2020, 2021, 2022, 2023]
  Fold 1: train <= 2018 (3297), val = 2019 (444)
  Fold 2: train <= 2019 (3741), val = 2020 (463)
  Fold 3: train <= 2020 (4204), val = 2021 (475)
  Fold 4: train <= 2021 (4679), val = 2022 (495)
  Fold 5: train <= 2022 (5174), val = 2023 (513)
Available years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Test: > 2023 (1028 samples)
Temporal k-fold (1 folds, expanding window). Val years: [2023]
  Fold 1: train <= 2022 (5174), val = 2023 (513)


/tmp/ipykernel_534/3206338114.py:92: RuntimeWarning: overflow encountered in expm1
  raw_ratio = np.sign(sym_lab) * np.expm1(np.abs(sym_lab))


Train: 1738 ({2010: np.int64(112), 2011: np.int64(114), 2012: np.int64(112), 2013: np.int64(116), 2014: np.int64(118), 2015: np.int64(123), 2016: np.int64(130), 2017: np.int64(135), 2018: np.int64(146), 2019: np.int64(150), 2020: np.int64(156), 2021: np.int64(160), 2022: np.int64(166)})
Test:  344 ({2024: np.int64(175), 2025: np.int64(169)})
Pipeline setup complete.


## Frequency-aware feature selection (for MultiFreqLSTM only)

Daily, monthly, and quarterly features, identified by their non-null cadence.

In [5]:
DAILY_FEATURES = [
    'Close_logret', 'Volume_log', 'DE10YT_Yield', 'IT10YT_Yield',
    'EURUSD_Close', 'EURUSD_Volume', 'FTMIB_Close', 'FTMIB_Volume',
    'GVZ_Close', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'VIX_Close',
    'Gold_Close', 'YC__level', 'YC__slope', 'YC__curv', 'ff_HML', 'ff_Mom',
    'TTF_Gas_Close', 'NFCI',
]
MONTHLY_FEATURES = ['Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM']
QUARTERLY_FEATURES = ['IT_GDP']

print(f"Daily ({len(DAILY_FEATURES)}): {DAILY_FEATURES}")
print(f"Monthly ({len(MONTHLY_FEATURES)}): {MONTHLY_FEATURES}")
print(f"Quarterly ({len(QUARTERLY_FEATURES)}): {QUARTERLY_FEATURES}")

daily_indices     = [feature_cols.index(f) for f in DAILY_FEATURES]
monthly_indices   = [feature_cols.index(f) for f in MONTHLY_FEATURES]
quarterly_indices = [feature_cols.index(f) for f in QUARTERLY_FEATURES]

Daily (21): ['Close_logret', 'Volume_log', 'DE10YT_Yield', 'IT10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume', 'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'VIX_Close', 'Gold_Close', 'YC__level', 'YC__slope', 'YC__curv', 'ff_HML', 'ff_Mom', 'TTF_Gas_Close', 'NFCI']
Monthly (5): ['Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM']
Quarterly (1): ['IT_GDP']


## Model Definitions

In [6]:
class ShallowLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_targets=3, dropout=0.2, static_dim=0):
        super().__init__()
        self.static_dim = static_dim
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size + static_dim, num_targets)

    def forward(self, x, lengths, static=None):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        if self.static_dim > 0 and static is not None:
            hidden = torch.cat([hidden, static.to(hidden.device, hidden.dtype)], dim=-1)
        logits = self.head(hidden)
        if logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        return logits

class StackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, num_targets=3,
                 dropout=0.2, static_dim=0):
        super().__init__()
        self.static_dim = static_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(hidden_size + static_dim, num_targets)

    def forward(self, x, lengths, static=None):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        if self.static_dim > 0 and static is not None:
            hidden = torch.cat([hidden, static.to(hidden.device, hidden.dtype)], dim=-1)
        logits = self.head(hidden)
        if logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        return logits

class MultiFreqLSTM(nn.Module):
    """Three-stream model matching lstm_multifreq.ipynb exactly.
    Daily:   LSTM(input=n_daily,  hidden=64, layers=2)
    Monthly: LSTM(input=n_monthly, hidden=32, layers=1)
    Quarterly: Linear(n_quarterly*4, 8) + ReLU  (NOT an LSTM)"""
    def __init__(self, n_daily, n_monthly, n_quarterly, n_static,
                 d_daily=64, d_monthly=32, d_quarterly=8,
                 n_layers_daily=2, n_layers_monthly=1,
                 num_targets=3, dropout=0.1):
        super().__init__()
        self.daily = nn.LSTM(n_daily, d_daily, num_layers=n_layers_daily,
                            batch_first=True, dropout=0.0)
        self.monthly = nn.LSTM(n_monthly, d_monthly, num_layers=n_layers_monthly,
                              batch_first=True, dropout=0.0)
        self.quarterly_proj = nn.Linear(n_quarterly * 4, d_quarterly)
        self.merge_dim = d_daily + d_monthly + d_quarterly + n_static
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.merge_dim, num_targets)
        self._squeeze = (num_targets == 1)

    def forward(self, daily, monthly, quarterly, lengths, static):
        packed_d = pack_padded_sequence(daily, lengths.cpu(),
                                        batch_first=True, enforce_sorted=True)
        _, (h_d, _) = self.daily(packed_d)
        h_d = h_d[-1]

        B = monthly.size(0)
        lengths_m = torch.full((B,), 12, dtype=torch.long, device=monthly.device)
        packed_m = pack_padded_sequence(monthly, lengths_m.cpu(),
                                        batch_first=True, enforce_sorted=False)
        _, (h_m, _) = self.monthly(packed_m)
        h_m = h_m[-1]

        q_flat = quarterly.reshape(B, -1)
        h_q = torch.relu(self.quarterly_proj(q_flat))

        h = torch.cat([h_d, h_m, h_q, static.to(h_d.device, h_d.dtype)], dim=-1)
        h = self.dropout(h)
        logits = self.head(h)
        if self._squeeze:
            logits = logits.squeeze(-1)
        return logits

selected_indices = [feature_cols.index(f) for f in (DAILY_FEATURES + MONTHLY_FEATURES + QUARTERLY_FEATURES)]

print("Models defined: ShallowLSTM, StackedLSTM, MultiFreqLSTM")

Models defined: ShallowLSTM, StackedLSTM, MultiFreqLSTM


## Datasets & DataLoaders

In [ ]:
def aggregate_to_cadence(window_dates, window_days, raw_window, target_indices, n_steps):
    """Aggregate a raw window to `n_steps` evenly-spaced bins (by day of year).
    raw_window: (T, F) array; target_indices: list of feature column indices to keep.
    Returns (n_steps, len(target_indices)) array, with NaN if no observations.
    """
    T = raw_window.shape[0]
    if T < 2:
        return np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    last_day = float(window_days[-1])
    if last_day <= 0:
        return np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    bin_size = last_day / n_steps
    out = np.full((n_steps, len(target_indices)), np.nan, dtype=np.float32)
    if len(target_indices) == 0:
        return out
    # Assign each timestep to its bin
    bin_idx = np.minimum((window_days / bin_size).astype(int), n_steps - 1)
    for f_i, f_idx in enumerate(target_indices):
        col = raw_window[:, f_idx]
        for b in range(n_steps):
            mask = bin_idx == b
            if mask.any():
                vals = col[mask]
                vals = vals[~np.isnan(vals)]
                if len(vals) > 0:
                    out[b, f_i] = vals[-1]  # take last non-NaN value (forward-fill)
    return out

def build_multifreq_window(raw_window, window_days, daily_idx, monthly_idx, quarterly_idx,
                            n_daily=252, n_monthly=12, n_quarterly=4):
    """Aggregate the raw window to the 3 cadences."""
    daily    = aggregate_to_cadence(None, window_days, raw_window, daily_idx,     n_daily)
    monthly  = aggregate_to_cadence(None, window_days, raw_window, monthly_idx,   n_monthly)
    quarterly = aggregate_to_cadence(None, window_days, raw_window, quarterly_idx, n_quarterly)
    # Forward-fill along the time axis (within stream)
    for arr in (daily, monthly, quarterly):
        for f in range(arr.shape[1]):
            last = np.nan
            for t in range(arr.shape[0]):
                if np.isnan(arr[t, f]):
                    arr[t, f] = last
                else:
                    last = arr[t, f]
            # Backward-fill leading NaNs
            if np.isnan(arr[0, f]):
                first_valid = np.nan
                for t in range(arr.shape[0]):
                    if not np.isnan(arr[t, f]):
                        first_valid = arr[t, f]
                        break
                arr[:, f] = first_valid
    # Replace any remaining NaN with 0
    daily    = np.nan_to_num(daily,    nan=0.0).astype(np.float32)
    monthly  = np.nan_to_num(monthly,  nan=0.0).astype(np.float32)
    quarterly = np.nan_to_num(quarterly, nan=0.0).astype(np.float32)
    return daily, monthly, quarterly

class YoYDatasetMTL(Dataset):
    def __init__(self, data_df, scaler, feature_indices, window_size=None):
        self.data_df = data_df.reset_index(drop=True)
        self.scaler = scaler
        self.feature_indices = feature_indices
        self.window_size = window_size

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.window_size is not None:
            window = window[-self.window_size:]
        if self.scaler is not None:
            window = self.scaler.transform(window)
            # Sanitize: RobustScaler can produce NaN/Inf (e.g. zero IQR).
            window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        static = row["static_data"].astype(np.float32)
        static = np.nan_to_num(static, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return (
            torch.from_numpy(window),
            torch.from_numpy(static),
            torch.tensor(target_clean, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            len(window),
        )

def collate_fn_mtl(batch):
    batch.sort(key=lambda x: x[4], reverse=True)
    sequences = [x[0] for x in batch]
    statics = torch.stack([x[1] for x in batch])
    targets = torch.stack([x[2] for x in batch])
    masks = torch.stack([x[3] for x in batch])
    lengths = torch.tensor([x[4] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, statics, lengths

class MultiFreqDataset(Dataset):
    def __init__(self, mtl_df, daily_scaler, monthly_scaler, quarterly_scaler, stat_scaler,
                 daily_idx, monthly_idx, quarterly_idx, n_daily=252, n_monthly=12, n_quarterly=4):
        self.mtl_df = mtl_df.reset_index(drop=True)
        self.daily_scaler = daily_scaler
        self.monthly_scaler = monthly_scaler
        self.quarterly_scaler = quarterly_scaler
        self.stat_scaler = stat_scaler
        self.daily_idx = daily_idx
        self.monthly_idx = monthly_idx
        self.quarterly_idx = quarterly_idx
        self.n_daily = n_daily
        self.n_monthly = n_monthly
        self.n_quarterly = n_quarterly

    def __len__(self):
        return len(self.mtl_df)

    def __getitem__(self, idx):
        row = self.mtl_df.iloc[idx]
        raw_window = row["window_data"]
        window_days = row["window_days"]
        d, m, q = build_multifreq_window(
            raw_window, window_days,
            self.daily_idx, self.monthly_idx, self.quarterly_idx,
            n_daily=self.n_daily, n_monthly=self.n_monthly, n_quarterly=self.n_quarterly,
        )
        if self.daily_scaler is not None:
            d = self.daily_scaler.transform(d)
            d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.monthly_scaler is not None:
            m = self.monthly_scaler.transform(m)
            m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.quarterly_scaler is not None:
            q = self.quarterly_scaler.transform(q)
            q = np.nan_to_num(q, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if self.stat_scaler is not None:
            static = self.stat_scaler.transform(
                row["static_data"].reshape(1, -1)
            ).reshape(-1).astype(np.float32)
        else:
            static = row["static_data"].astype(np.float32)
        static = np.nan_to_num(static, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        target = row["label_value"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return (
            torch.from_numpy(d.astype(np.float32)),
            torch.from_numpy(m.astype(np.float32)),
            torch.from_numpy(q.astype(np.float32)),
            torch.from_numpy(static),
            torch.tensor(target_clean, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            self.n_daily,
        )

def collate_fn_multifreq(batch):
    batch.sort(key=lambda x: x[6], reverse=True)
    ds = torch.stack([x[0] for x in batch])
    ms = torch.stack([x[1] for x in batch])
    qs = torch.stack([x[2] for x in batch])
    statics = torch.stack([x[3] for x in batch])
    targets = torch.stack([x[4] for x in batch])
    masks = torch.stack([x[5] for x in batch])
    lengths = torch.tensor([x[6] for x in batch])
    return ds, ms, qs, targets, masks, statics, lengths

print("Datasets ready.")

Datasets ready.


## Training Recipe with Regularization

In [ ]:
def symlog_inverse(arr):
    arr = np.clip(arr, -SYMLOG_CAP, SYMLOG_CAP)
    return np.sign(arr) * np.expm1(np.abs(arr))

def train_and_eval_mtl(model, train_loader, val_loader, optimizer,
                       trial=None, step_offset=0):
    """L1 loss, ES, LR scheduler, grad_clip=1.0, best_state checkpointing."""
    criterion = nn.L1Loss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    best_state = None
    best_metric = float("inf")
    epochs_no_improve = 0
    best_epoch = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            bx, by, bmask, bstatic, lengths = batch
            bx, by, bmask, bstatic = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE), bstatic.to(DEVICE)
            logits = model(bx, lengths, bstatic)
            loss_m = criterion(logits, by)
            loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0 * loss_m.sum()
            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        model.eval()
        all_preds, all_targets, all_masks = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                bx, by, bmask, bstatic, lengths = batch
                bx, bstatic = bx.to(DEVICE), bstatic.to(DEVICE)
                logits = model(bx, lengths, bstatic)
                all_masks.append(bmask.numpy())
                all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())
        if not all_preds:
            break
        y_pred = np.vstack(all_preds)
        y_true = np.vstack(all_targets)
        mask = np.vstack(all_masks)
        valid_sum, count = 0.0, 0
        for i in range(len(PREDICTION_TARGETS)):
            m = mask[:, i]
            if m.sum() == 0:
                continue
            valid_sum += np.mean(np.abs(symlog_inverse(y_pred[m, i]) - symlog_inverse(y_true[m, i])))
            count += 1
        current_metric = valid_sum / max(1, count)
        scheduler.step(current_metric)
        
        if np.isfinite(current_metric) and current_metric < best_metric:
            best_metric = current_metric
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if trial is not None:
            trial.report(current_metric, step_offset + epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        if epochs_no_improve >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric, best_epoch

def evaluate_mtl_loader(model, loader):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for bx, by, bmask, bstatic, lengths in loader:
            logits = model(bx.to(DEVICE), lengths, bstatic.to(DEVICE))
            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())
            all_masks.append(bmask.numpy())
    if not all_preds:
        return None
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks).astype(bool)
    y_pred_r = symlog_inverse(y_pred)
    y_true_r = symlog_inverse(y_true)
    res = {}
    all_err = []
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = float(np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i])))
        all_err.extend(np.abs(y_pred_r[m, i] - y_true_r[m, i]).tolist())
    res["Overall"] = float(np.mean(all_err)) if all_err else None
    return res

print("MTL training/eval helpers ready.")

def collect_predictions_mtl(model, loader):
    """Run model on loader, return (predictions, targets) in real space."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bx, by, bmask, bstatic, lengths in loader:
            logits = model(bx.to(DEVICE), lengths, bstatic.to(DEVICE))
            preds.append(logits.cpu().numpy())
            targets.append(by.numpy())
    yp = np.vstack(preds); yt = np.vstack(targets)
    yp_r = symlog_inverse(yp); yt_r = symlog_inverse(yt)
    return yp_r, yt_r

def collect_predictions_mf(model, loader):
    """MultiFreq version: run model on loader, return (predictions, targets) in real space."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bd, bm, bq, by, bmask, bstatic, lengths in loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            bstatic = bstatic.to(DEVICE)
            logits = model(bd, bm, bq, lengths, bstatic)
            preds.append(logits.cpu().numpy())
            targets.append(by.numpy())
    yp = np.vstack(preds); yt = np.vstack(targets)
    yp_r = symlog_inverse(yp); yt_r = symlog_inverse(yt)
    return yp_r, yt_r

print("collect_predictions helpers ready.")

MTL training/eval helpers ready.
collect_predictions helpers ready.


## MultiFreq-specific training/eval helpers

In [ ]:
def train_and_eval_mf(model, train_loader, val_loader, optimizer,
                      trial=None, step_offset=0):
    criterion = nn.L1Loss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    best_state = None
    best_metric = float("inf")
    best_epoch = 0
    epochs_no_improve = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for bd, bm, bq, by, bmask, bstatic, lengths in train_loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            by = by.to(DEVICE); bmask = bmask.to(DEVICE); bstatic = bstatic.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(bd, bm, bq, lengths, bstatic)
            loss_m = criterion(logits, by)
            loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0 * loss_m.sum()
            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        model.eval()
        all_preds, all_targets, all_masks = [], [], []
        with torch.no_grad():
            for bd, bm, bq, by, bmask, bstatic, lengths in val_loader:
                bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
                bstatic = bstatic.to(DEVICE)
                logits = model(bd, bm, bq, lengths, bstatic)
                all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())
                all_masks.append(bmask.numpy())
        if not all_preds:
            break
        y_pred = np.vstack(all_preds)
        y_true = np.vstack(all_targets)
        mask = np.vstack(all_masks).astype(bool)
        y_pred_r = symlog_inverse(y_pred)
        y_true_r = symlog_inverse(y_true)
        valid_sum, count = 0.0, 0
        for i in range(len(PREDICTION_TARGETS)):
            m = mask[:, i]
            if m.sum() == 0:
                continue
            valid_sum += np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i]))
            count += 1
        current_metric = valid_sum / max(1, count)
        scheduler.step(current_metric)

        if np.isfinite(current_metric) and current_metric < best_metric:
            best_metric = current_metric
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if trial is not None:
            trial.report(current_metric, step_offset + epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        if epochs_no_improve >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric, best_epoch

def evaluate_mf_loader(model, loader):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for bd, bm, bq, by, bmask, bstatic, lengths in loader:
            bd = bd.to(DEVICE); bm = bm.to(DEVICE); bq = bq.to(DEVICE)
            bstatic = bstatic.to(DEVICE)
            logits = model(bd, bm, bq, lengths, bstatic)
            all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())
            all_masks.append(bmask.numpy())
    if not all_preds:
        return None
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks).astype(bool)
    y_pred_r = symlog_inverse(y_pred)
    y_true_r = symlog_inverse(y_true)
    res = {}
    all_err = []
    for i, t in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue
        res[t] = float(np.mean(np.abs(y_pred_r[m, i] - y_true_r[m, i])))
        all_err.extend(np.abs(y_pred_r[m, i] - y_true_r[m, i]).tolist())
    res["Overall"] = float(np.mean(all_err)) if all_err else None
    return res

print("MultiFreq helpers ready.")

MultiFreq helpers ready.


## Loader builders per model

In [10]:
def build_mtl_loaders_for_fold(tr_df, va_df, params, feature_indices, static_dim):
    """Build train/val/test loaders for MTL (Shallow or Stacked LSTM).
    Returns (model, train_loader, val_loader, test_loader, scaler, stat_scaler)."""
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    te_mtl = extract_mtl_df(test_data)
    temp_sc = make_robust_scaler(tr_mtl, feature_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)
    te_mtl_s = apply_static_scaler(te_mtl, stat_sc, NEW_STATIC_SLICE)
    tr_ds = YoYDatasetMTL(tr_mtl_s, temp_sc, feature_indices)
    va_ds = YoYDatasetMTL(va_mtl_s, temp_sc, feature_indices)
    te_ds = YoYDatasetMTL(te_mtl_s, temp_sc, feature_indices)
    tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_mtl)
    va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_mtl)
    te_loader = DataLoader(te_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_mtl)
    return tr_loader, va_loader, te_loader, temp_sc, stat_sc

def make_shallow_mtl(tr_df, va_df, params, feature_indices):
    tr_loader, va_loader, te_loader, _, _ = build_mtl_loaders_for_fold(
        tr_df, va_df, params, feature_indices, STATIC_DIM_IMP)
    model = ShallowLSTM(
        input_size=len(feature_indices),
        hidden_size=params['hidden_size'],
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    return model, tr_loader, va_loader, te_loader

def make_stacked_mtl(tr_df, va_df, params, feature_indices, num_layers=4):
    tr_loader, va_loader, te_loader, _, _ = build_mtl_loaders_for_fold(
        tr_df, va_df, params, feature_indices, STATIC_DIM_IMP)
    model = StackedLSTM(
        input_size=len(feature_indices),
        hidden_size=params['hidden_size'],
        num_layers=num_layers,
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)
    return model, tr_loader, va_loader, te_loader

print("MTL loader builders ready.")

MTL loader builders ready.


In [11]:
def make_multifreq_loaders_for_fold(tr_df, va_df, params):
    """Build train/val/test loaders for MultiFreqLSTM with per-stream scalers."""
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    te_mtl = extract_mtl_df(test_data)
    daily_sc = make_robust_scaler(tr_mtl, daily_indices)
    monthly_sc = make_robust_scaler(tr_mtl, monthly_indices)
    quarterly_sc = make_robust_scaler(tr_mtl, quarterly_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)
    te_mtl_s = apply_static_scaler(te_mtl, stat_sc, NEW_STATIC_SLICE)
    tr_ds = MultiFreqDataset(tr_mtl_s, daily_sc, monthly_sc, quarterly_sc, None,
                             daily_indices, monthly_indices, quarterly_indices)
    va_ds = MultiFreqDataset(va_mtl_s, daily_sc, monthly_sc, quarterly_sc, None,
                             daily_indices, monthly_indices, quarterly_indices)
    te_ds = MultiFreqDataset(te_mtl_s, daily_sc, monthly_sc, quarterly_sc, None,
                             daily_indices, monthly_indices, quarterly_indices)
    tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    te_loader = DataLoader(te_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq,
                           num_workers=2, pin_memory=True, persistent_workers=True)
    model = MultiFreqLSTM(
        n_daily=len(daily_indices),
        n_monthly=len(monthly_indices),
        n_quarterly=len(quarterly_indices),
        n_static=STATIC_DIM_IMP,
        d_daily=64, d_monthly=32, d_quarterly=8,
        n_layers_daily=params.get('n_layers_daily', 2),
        n_layers_monthly=params.get('n_layers_monthly', 1),
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.1),
    ).to(DEVICE)
    return model, tr_loader, va_loader, te_loader

print("MultiFreq loader builder ready.")

MultiFreq loader builder ready.


## Optuna Search (single-fold HPO on largest fold)

In [12]:
def get_hpo_objective_mtl(make_model_fn, feature_indices, n_trials=OPTUNA_TRIALS):
    """Generic Optuna objective for MTL models.
    **Single-fold HPO on the largest fold** (train <= 2022, val = 2023).
    This is ~5x faster than 5-fold HPO with negligible loss in best-params quality
    (the largest fold has the most data and is closest to the test period)."""
    # Use only the LARGEST fold (last in `folds`).
    fold_local = folds[-1]
    tr_df, va_df, val_year, _ = fold_local
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    temp_sc = make_robust_scaler(tr_mtl, feature_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)

    def objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        dropout = trial.suggest_float("dropout", 0.2, 0.5)
        params = dict(hidden_size=hidden_size, lr=lr, batch_size=batch_size,
                      weight_decay=weight_decay, dropout=dropout)
        tr_ds = YoYDatasetMTL(tr_mtl_s, temp_sc, feature_indices)
        va_ds = YoYDatasetMTL(va_mtl_s, temp_sc, feature_indices)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn_mtl)
        va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_mtl)
        model = make_model_fn(params, feature_indices)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        try:
            _, best_metric, _ = train_and_eval_mtl(
                model, tr_loader, va_loader, optimizer,
                trial=trial, step_offset=0,
            )
        except optuna.exceptions.TrialPruned:
            raise
        return best_metric
    return objective

def run_optuna(study_name, objective_fn, n_trials=OPTUNA_TRIALS):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=15)
    study = optuna.create_study(direction="minimize", study_name=study_name, pruner=pruner)
    study.optimize(objective_fn, n_trials=n_trials)
    return study

print("HPO infrastructure ready (single-fold mode).")

HPO infrastructure ready (single-fold mode).


## Model A: ShallowLSTM 

In [ ]:
selected_indices = [feature_cols.index(f) for f in (DAILY_FEATURES + MONTHLY_FEATURES + QUARTERLY_FEATURES)]
print(f"Using {len(selected_indices)} features for Shallow/Stacked LSTM input.")

def make_shallow_fn(params, fi):
    return ShallowLSTM(len(selected_indices), params['hidden_size'], len(PREDICTION_TARGETS),
                        params.get('dropout', 0.2), static_dim=STATIC_DIM_IMP).to(DEVICE)

_t0 = _time.time()
study_A = run_optuna(
    "shallow_reg",
    get_hpo_objective_mtl(make_shallow_fn, selected_indices),
    n_trials=OPTUNA_TRIALS,
)
print(f"\nShallowLSTM best val MAE: {study_A.best_value:.4f}")
print(f"Best params: {study_A.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

Using 27 features for Shallow/Stacked LSTM input.


[I 2026-07-30 23:43:18,413] A new study created in memory with name: shallow_reg
[I 2026-07-30 23:44:51,830] Trial 0 finished with value: 0.8748998641967773 and parameters: {'hidden_size': 32, 'lr': 0.0004936584380668546, 'batch_size': 16, 'weight_decay': 0.00018837940669447293, 'dropout': 0.3257293035241983}. Best is trial 0 with value: 0.8748998641967773.
[I 2026-07-30 23:45:36,327] Trial 1 finished with value: 0.8669527173042297 and parameters: {'hidden_size': 64, 'lr': 0.0021267332427051485, 'batch_size': 64, 'weight_decay': 1.927655360126814e-06, 'dropout': 0.383943632344549}. Best is trial 1 with value: 0.8669527173042297.
[I 2026-07-30 23:47:01,892] Trial 2 finished with value: 0.8728415966033936 and parameters: {'hidden_size': 32, 'lr': 0.00043717608139031387, 'batch_size': 32, 'weight_decay': 0.00020689018288070105, 'dropout': 0.3552421154812777}. Best is trial 1 with value: 0.8669527173042297.
[I 2026-07-30 23:48:32,086] Trial 3 finished with value: 0.8893724083900452 and par


ShallowLSTM best val MAE: 0.8623
Best params: {'hidden_size': 64, 'lr': 0.009205265785667846, 'batch_size': 64, 'weight_decay': 1.120070398730548e-06, 'dropout': 0.4970242687542047}
Optuna time: 16.9 min


## Model B: StackedLSTM (L=4)

In [ ]:
def make_stacked_fn(params, fi, num_layers=4):
    return StackedLSTM(
        input_size=len(selected_indices),
        hidden_size=params['hidden_size'],
        num_layers=num_layers,
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.2),
        static_dim=STATIC_DIM_IMP,
    ).to(DEVICE)

_t0 = _time.time()
study_B = run_optuna(
    "stacked4_reg",
    get_hpo_objective_mtl(make_stacked_fn, selected_indices),
    n_trials=OPTUNA_TRIALS,
)
print(f"\nStackedLSTM L=4 best val MAE: {study_B.best_value:.4f}")
print(f"Best params: {study_B.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

[I 2026-07-31 00:00:10,069] A new study created in memory with name: stacked4_reg
[I 2026-07-31 00:02:21,692] Trial 0 finished with value: 0.8640233874320984 and parameters: {'hidden_size': 128, 'lr': 0.001372904930284215, 'batch_size': 32, 'weight_decay': 0.0001303782070216121, 'dropout': 0.3389358616014047}. Best is trial 0 with value: 0.8640233874320984.
[I 2026-07-31 00:03:07,947] Trial 1 finished with value: 0.8722233176231384 and parameters: {'hidden_size': 128, 'lr': 0.005035416785635754, 'batch_size': 64, 'weight_decay': 2.1607056222013676e-05, 'dropout': 0.3263686127695347}. Best is trial 0 with value: 0.8640233874320984.
[I 2026-07-31 00:05:10,720] Trial 2 finished with value: 0.87313312292099 and parameters: {'hidden_size': 32, 'lr': 0.003451025785261305, 'batch_size': 16, 'weight_decay': 2.5972646857959755e-05, 'dropout': 0.22361572026204815}. Best is trial 0 with value: 0.8640233874320984.
[I 2026-07-31 00:06:01,076] Trial 3 finished with value: 0.9019249081611633 and para


StackedLSTM L=4 best val MAE: 0.8632
Best params: {'hidden_size': 64, 'lr': 0.0029737332284412083, 'batch_size': 64, 'weight_decay': 3.4057472551096976e-05, 'dropout': 0.4399163768986402}
Optuna time: 25.2 min


## Model C: MultiFreqLSTM

In [ ]:
OPTUNA_TRIALS = 8

def make_mf_for_fold(params, tr_df, va_df):
    return make_multifreq_loaders_for_fold(tr_df, va_df, params)

def make_mf_fn_for_optuna(params):
    model = MultiFreqLSTM(
        n_daily=len(daily_indices),
        n_monthly=len(monthly_indices),
        n_quarterly=len(quarterly_indices),
        n_static=STATIC_DIM_IMP,
        d_daily=64, d_monthly=32, d_quarterly=8,
        n_layers_daily=params.get('n_layers_daily', 2),
        n_layers_monthly=params.get('n_layers_monthly', 1),
        num_targets=len(PREDICTION_TARGETS),
        dropout=params.get('dropout', 0.1),
    ).to(DEVICE)
    return model

def get_hpo_objective_mf():
    """MultiFreq Optuna objective — single fold (largest, train <= 2022, val = 2023)."""
    fold_local = folds[-1]
    tr_df, va_df, val_year, _ = fold_local
    tr_mtl = extract_mtl_df(tr_df)
    va_mtl = extract_mtl_df(va_df)
    daily_sc = make_robust_scaler(tr_mtl, daily_indices)
    monthly_sc = make_robust_scaler(tr_mtl, monthly_indices)
    quarterly_sc = make_robust_scaler(tr_mtl, quarterly_indices)
    stat_sc = fit_static_scaler(tr_mtl, NEW_STATIC_SLICE)
    tr_mtl_s = apply_static_scaler(tr_mtl, stat_sc, NEW_STATIC_SLICE)
    va_mtl_s = apply_static_scaler(va_mtl, stat_sc, NEW_STATIC_SLICE)

    def objective(trial):
        params = dict(
            batch_size=trial.suggest_categorical("batch_size", [16, 32, 64]),
            lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            dropout=trial.suggest_float("dropout", 0.1, 0.5),
            n_layers_daily=2,  # fixed
            n_layers_monthly=1,  # fixed
        )
        tr_ds = MultiFreqDataset(tr_mtl_s, daily_sc, monthly_sc, quarterly_sc, None,
                                 daily_indices, monthly_indices, quarterly_indices)
        va_ds = MultiFreqDataset(va_mtl_s, daily_sc, monthly_sc, quarterly_sc, None,
                                 daily_indices, monthly_indices, quarterly_indices)
        tr_loader = DataLoader(tr_ds, batch_size=params['batch_size'], shuffle=True,  collate_fn=collate_fn_multifreq)
        va_loader = DataLoader(va_ds, batch_size=params['batch_size'], shuffle=False, collate_fn=collate_fn_multifreq)
        model = make_mf_fn_for_optuna(params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        try:
            _, best_metric, _ = train_and_eval_mf(
                model, tr_loader, va_loader, optimizer,
                trial=trial, step_offset=0,
            )
        except optuna.exceptions.TrialPruned:
            raise
        except Exception as e:
            # Catch any GPU/runtime error (cuDNN, NaN, etc.) so Optuna marks
            # this trial as bad (returns inf) and the study continues.
            print(f"Trial failed: {type(e).__name__}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return float('inf')
        return best_metric
    return objective

_t0 = _time.time()
study_C = run_optuna(
    "multifreq_reg",
    get_hpo_objective_mf(),
    n_trials=OPTUNA_TRIALS,
)
print(f"\nMultiFreqLSTM best val MAE: {study_C.best_value:.4f}")
print(f"Best params: {study_C.best_trial.params}")
print(f"Optuna time: {(_time.time()-_t0)/60:.1f} min")

[I 2026-07-31 14:32:54,586] A new study created in memory with name: multifreq_reg
[I 2026-07-31 15:05:31,778] Trial 0 finished with value: 0.8839097023010254 and parameters: {'batch_size': 32, 'lr': 0.000310371864873874, 'weight_decay': 3.0382558829726287e-05, 'dropout': 0.30228500784047996}. Best is trial 0 with value: 0.8839097023010254.
[I 2026-07-31 15:25:39,669] Trial 1 finished with value: 0.8878691792488098 and parameters: {'batch_size': 32, 'lr': 0.0036213336795681283, 'weight_decay': 0.000616859345207868, 'dropout': 0.18869165674593977}. Best is trial 0 with value: 0.8839097023010254.
[I 2026-07-31 16:05:35,716] Trial 2 finished with value: 0.9083163142204285 and parameters: {'batch_size': 64, 'lr': 0.0003617613289064042, 'weight_decay': 2.5464917316558714e-06, 'dropout': 0.4668248047847935}. Best is trial 0 with value: 0.8839097023010254.
[I 2026-07-31 17:50:03,833] Trial 3 finished with value: 0.8626505732536316 and parameters: {'batch_size': 64, 'lr': 0.0001985341055806483

## Final Test: 5-seed run on test (2024+2025) for each model

In [13]:
def run_n_seeded_final_mtl(make_model_fn, params, n_seeds=N_FINAL_SEEDS):
    """Train `n_seeds` independent models on train/val (train<=2022, val=2023), test on 2024+2025."""
    results = {t: [] for t in PREDICTION_TARGETS + ["Overall"]}
    histories = []
    best_state_dicts = []  # store best seed (lowest test Overall MAE)
    test_res_per_seed = []
    for s in range(n_seeds):
        np.random.seed(SEED + s)
        torch.manual_seed(SEED + s)
        model, tr_loader, va_loader, te_loader = make_model_fn(params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        model, best_val, best_epoch = train_and_eval_mtl(model, tr_loader, va_loader, optimizer)
        test_res = evaluate_mtl_loader(model, te_loader)
        for k, v in test_res.items():
            results[k].append(v)
        test_res_per_seed.append(test_res)
    return results, test_res_per_seed

print("Multi-seed helper ready.")

Multi-seed helper ready.


In [ ]:
class _OptunaStudyStub:
    """Minimal Optuna study stub. Only exposes what downstream cells use:
    .best_value (float) and .best_trial.params (dict)."""
    def __init__(self, best_value, best_params):
        self.best_value = float(best_value)
        self.best_trial = type('TrialStub', (), {'params': dict(best_params)})()

_OPTUNA_RESULTS_A = {
    'best_value': float('nan'),
    'best_params': {
        'hidden_size': 64, 'lr': 0.009205265785667846, 'batch_size': 64,
        'weight_decay': 1.120070398730548e-06, 'dropout': 0.4970242687542047,
    },
}
_OPTUNA_RESULTS_B = {
    'best_value': float('nan'),
    'best_params': {
        'hidden_size': 64, 'lr': 0.0029737332284412083, 'batch_size': 64,
        'weight_decay': 3.4057472551096976e-05, 'dropout': 0.4399163768986402,
    },
}
_OPTUNA_RESULTS_C = {
    'best_value': 0.8626505732536316,
    'best_params': {
        'batch_size': 64, 'lr': 0.0001985341055806483,
        'weight_decay': 0.0003277040360791605, 'dropout': 0.1676639003470909,
    },
}

study_A = _OptunaStudyStub(_OPTUNA_RESULTS_A['best_value'], _OPTUNA_RESULTS_A['best_params'])
study_B = _OptunaStudyStub(_OPTUNA_RESULTS_B['best_value'], _OPTUNA_RESULTS_B['best_params'])
study_C = _OptunaStudyStub(_OPTUNA_RESULTS_C['best_value'], _OPTUNA_RESULTS_C['best_params'])
params_A = _OPTUNA_RESULTS_A['best_params']
params_B = _OPTUNA_RESULTS_B['best_params']
params_C = _OPTUNA_RESULTS_C['best_params']
print("=== HARDCODED Optuna results loaded ===")
print(f"ShallowLSTM   best params: {params_A}")
print(f"StackedLSTM L=4 best params: {params_B}")
print(f"MultiFreqLSTM  best val MAE = {study_C.best_value:.4f}")
print(f"MultiFreqLSTM  best params:  {params_C}")


=== HARDCODED Optuna results loaded ===
ShallowLSTM   best params: {'hidden_size': 64, 'lr': 0.009205265785667846, 'batch_size': 64, 'weight_decay': 1.120070398730548e-06, 'dropout': 0.4970242687542047}
StackedLSTM L=4 best params: {'hidden_size': 64, 'lr': 0.0029737332284412083, 'batch_size': 64, 'weight_decay': 3.4057472551096976e-05, 'dropout': 0.4399163768986402}
MultiFreqLSTM  best val MAE = 0.8627
MultiFreqLSTM  best params:  {'batch_size': 64, 'lr': 0.0001985341055806483, 'weight_decay': 0.0003277040360791605, 'dropout': 0.1676639003470909}


In [ ]:
params_A = study_A.best_trial.params
print(f"ShallowLSTM final params: {params_A}")
def make_A(params):
    return make_shallow_mtl(train_data, val_data, params, selected_indices)
_t0 = _time.time()
results_A, test_res_A = run_n_seeded_final_mtl(make_A, params_A, n_seeds=N_FINAL_SEEDS)
print(f"\nShallowLSTM final test (n={N_FINAL_SEEDS} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_A[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

ShallowLSTM final params: {'hidden_size': 64, 'lr': 0.009205265785667846, 'batch_size': 64, 'weight_decay': 1.120070398730548e-06, 'dropout': 0.4970242687542047}

ShallowLSTM final test (n=5 seeds):
  EBITDA      : 0.5279 ± 0.0100
  Net_Income  : 0.8608 ± 0.0087
  ROA         : 0.8265 ± 0.0073
  Overall     : 0.7383 ± 0.0083
Time: 3.1 min


In [ ]:
params_B = study_B.best_trial.params
print(f"StackedLSTM L=4 final params: {params_B}")
def make_B(params):
    return make_stacked_mtl(train_data, val_data, params, selected_indices, num_layers=4)
_t0 = _time.time()
results_B, test_res_B = run_n_seeded_final_mtl(make_B, params_B, n_seeds=N_FINAL_SEEDS)
print(f"\nStackedLSTM L=4 final test (n={N_FINAL_SEEDS} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_B[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

StackedLSTM L=4 final params: {'hidden_size': 64, 'lr': 0.0029737332284412083, 'batch_size': 64, 'weight_decay': 3.4057472551096976e-05, 'dropout': 0.4399163768986402}

StackedLSTM L=4 final test (n=5 seeds):
  EBITDA      : 0.5184 ± 0.0057
  Net_Income  : 0.8528 ± 0.0061
  ROA         : 0.8169 ± 0.0024
  Overall     : 0.7293 ± 0.0041
Time: 5.9 min


In [ ]:
def run_n_seeded_final_mf(params, n_seeds=N_FINAL_SEEDS):
    """Train `n_seeds` independent MultiFreqLSTM models, evaluate on test set."""
    results = {t: [] for t in PREDICTION_TARGETS + ["Overall"]}
    for s in range(n_seeds):
        np.random.seed(SEED + s); torch.manual_seed(SEED + s)
        model, tr_loader, va_loader, te_loader = make_multifreq_loaders_for_fold(
            train_data, val_data, params)
        optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        model, _, _ = train_and_eval_mf(model, tr_loader, va_loader, optimizer)
        test_res = evaluate_mf_loader(model, te_loader)
        for k, v in test_res.items():
            results[k].append(v)
    return results

params_C = study_C.best_trial.params
print(f"MultiFreqLSTM final params: {params_C}")
N_FINAL_SEEDS_C = 2
_t0 = _time.time()
results_C = run_n_seeded_final_mf(params_C, n_seeds=N_FINAL_SEEDS_C)
print(f"\nMultiFreqLSTM final test (n={N_FINAL_SEEDS_C} seeds):")
for k in PREDICTION_TARGETS + ["Overall"]:
    vals = results_C[k]
    print(f"  {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

MultiFreqLSTM final params: {'batch_size': 64, 'lr': 0.0001985341055806483, 'weight_decay': 0.0003277040360791605, 'dropout': 0.1676639003470909}

MultiFreqLSTM final test (n=2 seeds):
  EBITDA      : 0.5384 ± 0.0007
  Net_Income  : 0.8971 ± 0.0108
  ROA         : 0.8533 ± 0.0146
  Overall     : 0.7629 ± 0.0087
Time: 83.7 min


## Per-Fold Evaluation: K=3 seeds × 5 folds

In [18]:
N_SEEDS_PER_FOLD = 3

def run_per_fold_final_mtl(make_model_fn, params, n_seeds=N_SEEDS_PER_FOLD, collect_preds=True):
    """For each fold, train K seeds with the best Optuna params and evaluate on test set.
    Returns a nested dict: per_fold[fold_idx] = {
        'val_year': int, 'seed_maes': [list of per-seed test MAE dicts],
        'preds_untrained': {seed_idx: array (real space)},
        'preds_trained':   {seed_idx: array (real space)},
        'targets':          array (real space),
    }"""
    per_fold = {}
    for fold_idx, (tr_df, va_df, val_year, fi) in enumerate(folds):
        seed_maes = []
        preds_u_dict = {}
        preds_t_dict = {}
        targets_arr = None
        for s in range(n_seeds):
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            # Untrained baseline predictions
            model_u, tr_loader_u, va_loader_u, te_loader_u = make_model_fn(params, tr_df, va_df)
            yp_u, yt = collect_predictions_mtl(model_u, te_loader_u) if collect_preds else (None, None)
            del model_u
            # Train
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model, tr_loader, va_loader, te_loader = make_model_fn(params, tr_df, va_df)
            optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
            model, _, _ = train_and_eval_mtl(model, tr_loader, va_loader, optimizer)
            test_res = evaluate_mtl_loader(model, te_loader)
            seed_maes.append(test_res)
            if collect_preds:
                yp_t, yt_chk = collect_predictions_mtl(model, te_loader)
                targets_arr = yt_chk
                preds_u_dict[s] = yp_u
                preds_t_dict[s] = yp_t
            del model
        per_fold[fold_idx] = {
            'val_year': int(val_year),
            'seed_maes': seed_maes,
            'preds_untrained': preds_u_dict,
            'preds_trained':   preds_t_dict,
            'targets':          targets_arr,
        }
        maes_overall = [m['Overall'] for m in seed_maes]
        print(f"  Fold {fi} (val={val_year}): "
              f"Overall MAE = {np.mean(maes_overall):.4f} ± {np.std(maes_overall):.4f}")
    return per_fold

print("Per-fold MTL helper ready.")

Per-fold MTL helper ready.


In [19]:
def run_per_fold_final_mf(params, n_seeds=N_SEEDS_PER_FOLD, collect_preds=True):
    """MultiFreq version of per-fold evaluation."""
    per_fold = {}
    for fold_idx, (tr_df, va_df, val_year, fi) in enumerate(folds):
        seed_maes = []
        preds_u_dict = {}
        preds_t_dict = {}
        targets_arr = None
        for s in range(n_seeds):
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model_u, _, _, te_loader_u = make_multifreq_loaders_for_fold(tr_df, va_df, params)
            yp_u, yt = collect_predictions_mf(model_u, te_loader_u) if collect_preds else (None, None)
            del model_u
            np.random.seed(SEED + s); torch.manual_seed(SEED + s)
            model, tr_loader, va_loader, te_loader = make_multifreq_loaders_for_fold(
                tr_df, va_df, params)
            optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
            model, _, _ = train_and_eval_mf(model, tr_loader, va_loader, optimizer)
            test_res = evaluate_mf_loader(model, te_loader)
            seed_maes.append(test_res)
            if collect_preds:
                yp_t, yt_chk = collect_predictions_mf(model, te_loader)
                targets_arr = yt_chk
                preds_u_dict[s] = yp_u
                preds_t_dict[s] = yp_t
            del model
        per_fold[fold_idx] = {
            'val_year': int(val_year),
            'seed_maes': seed_maes,
            'preds_untrained': preds_u_dict,
            'preds_trained':   preds_t_dict,
            'targets':          targets_arr,
        }
        maes_overall = [m['Overall'] for m in seed_maes]
        print(f"  Fold {fi} (val={val_year}): "
              f"Overall MAE = {np.mean(maes_overall):.4f} ± {np.std(maes_overall):.4f}")
    return per_fold

print("Per-fold MF helper ready.")

Per-fold MF helper ready.


### Per-fold test (3 seeds × 5 folds) for each model

In [ ]:
def make_A_for_fold(params, tr_df, va_df):
    return make_shallow_mtl(tr_df, va_df, params, selected_indices)

def make_B_for_fold(params, tr_df, va_df):
    return make_stacked_mtl(tr_df, va_df, params, selected_indices, num_layers=4)

print("Per-fold wrapper functions defined.")

Per-fold wrapper functions defined.


In [ ]:
print("ShallowLSTM per-fold test:")
_t0 = _time.time()
per_fold_A = run_per_fold_final_mtl(make_A_for_fold, params_A, n_seeds=2)  # 2 seeds for A
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

ShallowLSTM per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.8965 ± 0.0020
  Fold 2 (val=2020): Overall MAE = 0.7464 ± 0.0046
  Fold 3 (val=2021): Overall MAE = 0.7437 ± 0.0072
  Fold 4 (val=2022): Overall MAE = 0.7446 ± 0.0019
  Fold 5 (val=2023): Overall MAE = 0.7369 ± 0.0041
Time: 6.5 min


In [ ]:
print("StackedLSTM L=4 per-fold test:")
_t0 = _time.time()
per_fold_B = run_per_fold_final_mtl(make_B_for_fold, params_B, n_seeds=2)  # 2 seeds for B
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

StackedLSTM L=4 per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.7399 ± 0.0011
  Fold 2 (val=2020): Overall MAE = 0.7428 ± 0.0044
  Fold 3 (val=2021): Overall MAE = 0.7518 ± 0.0076
  Fold 4 (val=2022): Overall MAE = 0.7318 ± 0.0033
  Fold 5 (val=2023): Overall MAE = 0.7264 ± 0.0017
Time: 12.2 min


In [ ]:
print("MultiFreqLSTM per-fold test:")
_t0 = _time.time()
per_fold_C = run_per_fold_final_mf(params_C, n_seeds=1)  # 1 seed because takes longer
print(f"Time: {(_time.time()-_t0)/60:.1f} min")

MultiFreqLSTM per-fold test:
  Fold 1 (val=2019): Overall MAE = 0.7757 ± 0.0000
  Fold 2 (val=2020): Overall MAE = 0.7653 ± 0.0000
  Fold 3 (val=2021): Overall MAE = 0.7756 ± 0.0000
  Fold 4 (val=2022): Overall MAE = 0.8035 ± 0.0000
  Fold 5 (val=2023): Overall MAE = 0.7542 ± 0.0000
Time: 238.3 min


In [ ]:
SINGLE_SPLIT = {
    "ShallowLSTM (notebook 1)": {
        "n_seeds": 5,
        "EBITDA": (0.5279, 0.0100),
        "Net_Income": (0.8608, 0.0087),
        "ROA": (0.8265, 0.0073),
        "Overall": (0.7383, 0.0083),
    },
    "StackedLSTM L=4 (notebooks 2 & 4)": {
        "n_seeds": 5,
        "EBITDA": (0.5184, 0.0057),
        "Net_Income": (0.8528, 0.0061),
        "ROA": (0.8169, 0.0024),
        "Overall": (0.7293, 0.0041),
    },
    "MultiFreqLSTM (notebook 3)": {
        "n_seeds": 2,
        "EBITDA": (0.5384, 0.0007),
        "Net_Income": (0.8971, 0.0108),
        "ROA": (0.8533, 0.0146),
        "Overall": (0.7629, 0.0087),
    },
}

PER_FOLD = {
    "ShallowLSTM (notebook 1)": {2019: 0.8965, 2020: 0.7464, 2021: 0.7437, 2022: 0.7446, 2023: 0.7369},
    "StackedLSTM L=4 (notebooks 2 & 4)": {2019: 0.7399, 2020: 0.7428, 2021: 0.7518, 2022: 0.7318, 2023: 0.7264},
    "MultiFreqLSTM (notebook 3)": {2019: 0.7757, 2020: 0.7653, 2021: 0.7756, 2022: 0.8035, 2023: 0.7542},
}

OPTUNA_VAL = {
    "ShallowLSTM (notebook 1)": float('nan'),   # not recorded
    "StackedLSTM L=4 (notebooks 2 & 4)": float('nan'),
    "MultiFreqLSTM (notebook 3)": 0.8626505732536316,
}

BEST_PARAMS = {
    "ShallowLSTM (notebook 1)": {'hidden_size': 64, 'lr': 0.009205265785667846,
                                 'batch_size': 64, 'weight_decay': 1.120070398730548e-06,
                                 'dropout': 0.4970242687542047},
    "StackedLSTM L=4 (notebooks 2 & 4)": {'hidden_size': 64, 'lr': 0.0029737332284412083,
                                          'batch_size': 64, 'weight_decay': 3.4057472551096976e-05,
                                          'dropout': 0.4399163768986402},
    "MultiFreqLSTM (notebook 3)": {'batch_size': 64, 'lr': 0.0001985341055806483,
                                   'weight_decay': 0.0003277040360791605, 'dropout': 0.1676639003470909},
}

def pf_summary(pf):
    """Mean +/- std of per-fold means (std across the 5 fold means)."""
    means = list(pf.values())
    return f"{np.mean(means):.4f} ± {np.std(means):.4f}"

pf_str = {m: pf_summary(PER_FOLD[m]) for m in PER_FOLD}

print("Per-fold test MAE (Overall), per validation year (A/B = 2 seeds, C = 1 seed):")
for m, pf in PER_FOLD.items():
    years = sorted(pf.keys())
    print(f"  {m:28s}: " + ", ".join(f"{y}: {pf[y]:.4f}" for y in years))
print()
print("Single-split test MAE (Overall):")
for m, r in SINGLE_SPLIT.items():
    print(f"  {m:28s} (n={r['n_seeds']} seeds): {r['Overall'][0]:.4f} ± {r['Overall'][1]:.4f}")
print("\nAll numeric results are hardcoded; no training or plotting performed.")

Per-fold test MAE (Overall), per validation year (A/B = 2 seeds, C = 1 seed):
  ShallowLSTM (notebook 1)    : 2019: 0.8965, 2020: 0.7464, 2021: 0.7437, 2022: 0.7446, 2023: 0.7369
  StackedLSTM L=4 (notebooks 2 & 4): 2019: 0.7399, 2020: 0.7428, 2021: 0.7518, 2022: 0.7318, 2023: 0.7264
  MultiFreqLSTM (notebook 3)  : 2019: 0.7757, 2020: 0.7653, 2021: 0.7756, 2022: 0.8035, 2023: 0.7542

Single-split test MAE (Overall):
  ShallowLSTM (notebook 1)     (n=5 seeds): 0.7383 ± 0.0083
  StackedLSTM L=4 (notebooks 2 & 4) (n=5 seeds): 0.7293 ± 0.0041
  MultiFreqLSTM (notebook 3)   (n=2 seeds): 0.7629 ± 0.0087

All numeric results are hardcoded; no training or plotting performed.


## Final Comparison Table

In [ ]:
summary = pd.DataFrame([
    {
        "Model": m,
        "Best Optuna val MAE": OPTUNA_VAL[m],
        "Test MAE (EBITDA)": f"{SINGLE_SPLIT[m]['EBITDA'][0]:.4f} ± {SINGLE_SPLIT[m]['EBITDA'][1]:.4f}",
        "Test MAE (Net_Income)": f"{SINGLE_SPLIT[m]['Net_Income'][0]:.4f} ± {SINGLE_SPLIT[m]['Net_Income'][1]:.4f}",
        "Test MAE (ROA)": f"{SINGLE_SPLIT[m]['ROA'][0]:.4f} ± {SINGLE_SPLIT[m]['ROA'][1]:.4f}",
        "Test MAE (Overall)": f"{SINGLE_SPLIT[m]['Overall'][0]:.4f} ± {SINGLE_SPLIT[m]['Overall'][1]:.4f}",
        "Per-fold test MAE": pf_str[m],
        "Best params": str(BEST_PARAMS[m]),
    }
    for m in ["ShallowLSTM (notebook 1)", "StackedLSTM L=4 (notebooks 2 & 4)", "MultiFreqLSTM (notebook 3)"]
])
display(summary)

# Best model by single-split headline number
best_idx = summary["Test MAE (Overall)"].apply(lambda s: float(s.split(' ± ')[0])).idxmin()
print(f"\nBest model by single-split test Overall MAE: {summary.loc[best_idx, 'Model']}")
print(f"  Test MAE (Overall): {summary.loc[best_idx, 'Test MAE (Overall)']}")
print(f"  Per-fold test MAE:   {summary.loc[best_idx, 'Per-fold test MAE']}")

# Best model by per-fold test MAE
best_pf_idx = summary["Per-fold test MAE"].apply(lambda s: float(s.split(' ± ')[0])).idxmin()
print(f"\nBest model by per-fold test MAE: {summary.loc[best_pf_idx, 'Model']}")
print(f"  Per-fold test MAE:   {summary.loc[best_pf_idx, 'Per-fold test MAE']}")

,Model,Best Optuna val MAE,Test MAE (EBITDA),Test MAE (Net_Income),Test MAE (ROA),Test MAE (Overall),Per-fold test MAE,Best params
0,ShallowLSTM (notebook 1),NaN,0.5279 ± 0.0100,0.8608 ± 0.0087,0.8265 ± 0.0073,0.7383 ± 0.0083,0.7736 ± 0.0615,"{'hidden_size': 64, 'lr': 0.009205265785667846..."
1,StackedLSTM L=4 (notebooks 2 & 4),NaN,0.5184 ± 0.0057,0.8528 ± 0.0061,0.8169 ± 0.0024,0.7293 ± 0.0041,0.7385 ± 0.0088,"{'hidden_size': 64, 'lr': 0.002973733228441208..."
2,MultiFreqLSTM (notebook 3),0.862651,0.5384 ± 0.0007,0.8971 ± 0.0108,0.8533 ± 0.0146,0.7629 ± 0.0087,0.7749 ± 0.0164,"{'batch_size': 64, 'lr': 0.0001985341055806483..."



Best model by single-split test Overall MAE: StackedLSTM L=4 (notebooks 2 & 4)
  Test MAE (Overall): 0.7293 ± 0.0041
  Per-fold test MAE:   0.7385 ± 0.0088

Best model by per-fold test MAE: StackedLSTM L=4 (notebooks 2 & 4)
  Per-fold test MAE:   0.7385 ± 0.0088
